# Improving RAG with Reranking, Chunking, and Conversational Memory
## Step 1: Setup and Loading the Dataset

In [1]:
!pip install -q datasets sentence-transformers chromadb langchain-core langchain-text-splitters langchain-chroma langchain-huggingface "pandas==2.2.3"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [1]:
import re
import pandas as pd
from datasets import load_dataset
import chromadb, sentence_transformers, torch

print("pandas:", pd.__version__)
print("chromadb:", chromadb.__version__)
print("GPU available:", torch.cuda.is_available())

pandas: 2.2.3
chromadb: 1.5.9
GPU available: True


In [2]:
ds = load_dataset("ccdv/arxiv-summarization", "document", split="train[:100]")
print(ds)

README.md:   0%|          | 0.00/3.96k [00:00<?, ?B/s]

document/train-00000-of-00015.parquet: reconstructing file:   0%|          |  0.00B /  227MB            

document/train-00000-of-00015.parquet: downloading bytes:           |  0.00B            

document/train-00001-of-00015.parquet: reconstructing file:   0%|          |  0.00B /  226MB            

document/train-00001-of-00015.parquet: downloading bytes:           |  0.00B            

document/train-00002-of-00015.parquet: reconstructing file:   0%|          |  0.00B /  226MB            

... (29 outputs left out)

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

Dataset({
    features: ['article', 'abstract'],
    num_rows: 100
})


In [3]:
lengths = [len(r["article"]) for r in ds]
print("number of documents:", len(ds))
print("avg characters per article:", int(sum(lengths) / len(lengths)))
print("shortest:", min(lengths), " longest:", max(lengths))
print("contains newlines:", any("\n" in r["article"] for r in ds))

print("\nSample article (doc 7):\n", ds[7]["article"][:400])
print("\nIts abstract:\n", ds[7]["abstract"][:300])

number of documents: 100
avg characters per article: 36370
shortest: 3420  longest: 193700
contains newlines: False

Sample article (doc 7):
 with significant research efforts being directed to the development of neurocomputers based on the functionalities of the brain , a seismic shift is expected in the domain of computing based on the traditional von - neumann model . the @xmath0 @xcite , @xmath1 @xcite and the ibm @xmath2 @xcite are instances of recent flagship neuromorphic projects that aim to develop brain - inspired computing pla

Its abstract:
 synaptic memory is considered to be the main element responsible for learning and cognition in humans . 
 although traditionally non - volatile long - term plasticity changes have been implemented in nanoelectronic synapses for neuromorphic applications , recent studies in neuroscience have revealed


### Dataset Overview
Dataset: ccdv/arxiv-summarization (Hugging Face), **document subset**, **first 100 documents of the train split**. Each row has a full paper (article) and its summary (abstract); only the article text is indexed.
- 100 documents, average **36,370 characters** (shortest 3,420, longest 193,700). The papers are long and not pre-chunked.
- **No newlines**, so paragraph splitting is impossible; splitting must use sentence boundaries.
- Lowercased text with **LaTeX placeholders** (@xmath0, @xcite) and tokenized punctuation ("model , and", "von - neumann").

## Step 2: Preprocessing

In [4]:
def clean_text(t):
    t = re.sub(r"@xmath\d+", " ", t)
    t = re.sub(r"@xcite", " ", t)
    t = t.replace("``", '"').replace("''", '"')
    t = re.sub(r"\s+-\s+", "-", t)
    t = re.sub(r"\s+([.,;:?!)\]])", r"\1", t)    "
    t = re.sub(r"([(\[])\s+", r"\1", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

docs = []
for i, row in enumerate(ds):
    text = clean_text(row["article"])
    if len(text) > 500:
        docs.append({"doc_id": i, "text": text, "abstract": clean_text(row["abstract"])})

print("documents kept:", len(docs))

documents kept: 100


In [5]:
print("BEFORE:\n", ds[7]["article"][:400])
print("\nAFTER:\n", docs[7]["text"][:400])

raw_len = sum(len(r["article"]) for r in ds)
clean_len = sum(len(d["text"]) for d in docs)
print(f"\ntotal characters before: {raw_len:,}")
print(f"total characters after:  {clean_len:,}")

raw_xmath = sum(r["article"].count("@xmath") + r["article"].count("@xcite") for r in ds)
clean_xmath = sum(d["text"].count("@xmath") + d["text"].count("@xcite") for d in docs)
print(f"\n@xmath/@xcite placeholders before: {raw_xmath:,}")
print(f"@xmath/@xcite placeholders after:  {clean_xmath:,}")

BEFORE:
 with significant research efforts being directed to the development of neurocomputers based on the functionalities of the brain , a seismic shift is expected in the domain of computing based on the traditional von - neumann model . the @xmath0 @xcite , @xmath1 @xcite and the ibm @xmath2 @xcite are instances of recent flagship neuromorphic projects that aim to develop brain - inspired computing pla

AFTER:
 with significant research efforts being directed to the development of neurocomputers based on the functionalities of the brain, a seismic shift is expected in the domain of computing based on the traditional von-neumann model. the, and the ibm are instances of recent flagship neuromorphic projects that aim to develop brain-inspired computing platforms suitable for recognition (image, video, speec

total characters before: 3,637,064
total characters after:  3,122,643

@xmath/@xcite placeholders before: 41,187
@xmath/@xcite placeholders after:  0


### Preprocessing Summary
Removed @xmath / @xcite placeholders, fixed quotes, hyphens ("von - neumann" → "von-neumann") and punctuation spacing ("brain ," → "brain,"), and collapsed whitespace. All 100 documents kept.
- Placeholders: **41,187 → 0**
- Characters: **3,637,064 → 3,122,643** (~14% noise removed)

**Why:** noise tokens hurt embeddings; clean punctuation lets Strategy 2 detect sentence boundaries.
**Limitation:** math and some entity names were already replaced by placeholders in the dataset and cannot be recovered.

## Step 3: Chunking Strategies
### Strategy 1: Fixed-size chunking

In [6]:
from langchain_core.documents import Document

CHUNK_SIZE = 800
CHUNK_OVERLAP = 100

def fixed_size_chunks(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step) if text[i:i + size].strip()]

chunks_s1 = []
for d in docs:
    for j, c in enumerate(fixed_size_chunks(d["text"])):
        chunks_s1.append(Document(
            page_content=c,
            metadata={"doc_id": d["doc_id"], "chunk_id": f"s1-{d['doc_id']}-{j}", "strategy": "fixed"}
        ))

print("Strategy 1 chunks:", len(chunks_s1))
print("\nExample chunk:\n", chunks_s1[1].page_content)

Strategy 1 chunks: 4509

Example chunk:
 egularized kernel based models for additive models have been published when the focus is on sparsity and when the classical least squares loss function is used, see e.g.,,,,, and the references therein. of course, the least squares loss function is differentiable and has many nice mathematical properties, but it is only locally lipschitz continuous and therefore regularized kernel based methods based on this loss function typically suffer on bad statistical robustness properties, even if the kernel is bounded. this is in sharp contrast to kernel methods based on a lipschitz continuous loss function and on a bounded loss function, where results on upper bounds for the maxbias bias and on a bounded influence function are known, see e.g. for the general case and for additive models. therefore


### Strategy 2: Sentence-aware recursive chunking

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[". ", "? ", "! ", "; ", ", ", " ", ""],
    keep_separator="end",
)

chunks_s2 = []
for d in docs:
    for j, c in enumerate(splitter.split_text(d["text"])):
        chunks_s2.append(Document(
            page_content=c,
            metadata={"doc_id": d["doc_id"], "chunk_id": f"s2-{d['doc_id']}-{j}", "strategy": "sentence"}
        ))

print("Strategy 2 chunks:", len(chunks_s2))
print("\nExample chunk:\n", chunks_s2[1].page_content)

Strategy 2 chunks: 4687

Example chunk:
 in the last years many interesting results on learning rates of regularized kernel based models for additive models have been published when the focus is on sparsity and when the classical least squares loss function is used, see e.g.,,,,, and the references therein. of course, the least squares loss function is differentiable and has many nice mathematical properties, but it is only locally lipschitz continuous and therefore regularized kernel based methods based on this loss function typically suffer on bad statistical robustness properties, even if the kernel is bounded.


In [8]:
def chunk_stats(chunks, name):
    lens = [len(c.page_content) for c in chunks]
    ends_clean = sum(c.page_content.rstrip().endswith((".", "?", "!")) for c in chunks)
    return {
        "strategy": name,
        "num_chunks": len(chunks),
        "avg_chars": round(sum(lens) / len(lens)),
        "min_chars": min(lens),
        "max_chars": max(lens),
        "% ending at sentence boundary": round(100 * ends_clean / len(chunks), 1),
    }

chunk_table = pd.DataFrame([
    chunk_stats(chunks_s1, "S1: fixed-size"),
    chunk_stats(chunks_s2, "S2: sentence-aware"),
])
display(chunk_table)

,strategy,num_chunks,avg_chars,min_chars,max_chars,% ending at sentence boundary
0,S1: fixed-size,4509,790,5,800,3.5
1,S2: sentence-aware,4687,693,8,799,96.7


### Chunking Summary
Both strategies use **chunk size 800 characters, overlap 100**, so the only difference is where text is cut.

| Strategy | Chunks | Avg chars | Ends at sentence boundary |
|---|---|---|---|
| S1: Fixed-size | 4,509 | 790 | 3.5% |
| S2: Sentence-aware | 4,687 | 693 | 96.7% |

- **S1 (fixed-size):** cuts every 800 characters regardless of content, often mid-word or mid-sentence (e.g. a chunk starting "egularized…"), and leaves tiny tail chunks (min 5 chars).
- **S2 (sentence-aware):** RecursiveCharacterTextSplitter splits at sentence endings first, then clauses, then words, so chunks are complete sentences.

**Hypothesis:** S2 should retrieve more coherent, useful chunks.

## Step 4: Embedding and Indexing in ChromaDB

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = "BAAI/bge-small-en-v1.5"
embeddings = HuggingFaceEmbeddings(
    model_name=embed_model,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 64},
)

test = embeddings.embed_documents(["this is a test chunk", "another test chunk"])
print("embedding model:", embed_model)
print("vectors:", len(test), " dimensions:", len(test[0]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding model: BAAI/bge-small-en-v1.5
vectors: 2  dimensions: 384


In [10]:
from langchain_chroma import Chroma
from tqdm.auto import tqdm

def build_store(collection_name, chunks, batch_size=256):
    store = Chroma(
        collection_name=collection_name,
        embedding_function=embeddings,
        collection_metadata={"hnsw:space": "cosine"},   # cosine similarity
    )
    store.reset_collection()   # start empty, so re-running never creates duplicates
    for i in tqdm(range(0, len(chunks), batch_size), desc=collection_name):
        batch = chunks[i:i + batch_size]
        store.add_documents(documents=batch, ids=[c.metadata["chunk_id"] for c in batch])
    return store

store_s1 = build_store("arxiv_s1_fixed", chunks_s1)
store_s2 = build_store("arxiv_s2_sentence", chunks_s2)

print("S1 vectors stored:", store_s1._collection.count())
print("S2 vectors stored:", store_s2._collection.count())

arxiv_s1_fixed:   0%|          | 0/18 [00:00<?, ?it/s]

arxiv_s2_sentence:   0%|          | 0/19 [00:00<?, ?it/s]

S1 vectors stored: 4509
S2 vectors stored: 4687


In [11]:
test_q = "learning rates of kernel methods for additive models"
for name, store in [("S1", store_s1), ("S2", store_s2)]:
    top = store.similarity_search(test_q, k=1)[0]
    print(f"{name} top hit -> doc {top.metadata['doc_id']}: {top.page_content[:150]}...\n")

S1 top hit -> doc 0:  compare their risks evaluated for test data. for the same reason we will also not cover sparsity. consistency of support vector machines generated by...

S2 top hit -> doc 0: of course, a practical approach might be to fit both models and compare their risks evaluated for test data. for the same reason we will also not cove...



### Indexing Summary
- **Embedding model:** BAAI/bge-small-en-v1.5 (384-dim, normalized), on GPU.
- **Vector DB:** ChromaDB, cosine similarity, one collection per strategy:

| Collection | Strategy | Vectors stored |
|---|---|---|
| arxiv_s1_fixed | S1: Fixed-size | 4,509 |
| arxiv_s2_sentence | S2: Sentence-aware | 4,687 |

Each chunk stores metadata (doc_id, chunk_id, strategy) used for evaluation. A test search returns doc 0 for both collections.

## Step 5: Retrieval Without and With Reranking
### Sample queries
Four queries, each targeting a specific paper in the corpus. The same queries are used for both runs.

In [12]:
K = 5   # number of results we evaluate

queries = [
    {"qid": "Q1", "target_doc": 0,
     "query": "What learning rates do regularized kernel based methods achieve for additive models?"},
    {"qid": "Q2", "target_doc": 3,
     "query": "How does a random phase diffuser affect the scintillation of laser light?"},
    {"qid": "Q3", "target_doc": 28,
     "query": "How can quantum error correction protect against fully correlated noise?"},
    {"qid": "Q4", "target_doc": 32,
     "query": "How is the origin-destination matrix estimated from passenger counts and station surveys?"},
]

for q in queries:
    print(q["qid"], "| target doc", q["target_doc"], "|", q["query"])

Q1 | target doc 0 | What learning rates do regularized kernel based methods achieve for additive models?
Q2 | target doc 3 | How does a random phase diffuser affect the scintillation of laser light?
Q3 | target doc 28 | How can quantum error correction protect against fully correlated noise?
Q4 | target doc 32 | How is the origin-destination matrix estimated from passenger counts and station surveys?


### 5.1 Retrieval WITHOUT reranking

In [13]:
stores = {"S1": store_s1, "S2": store_s2}
results = {}

def retrieve(store, query, k=K):
    return store.similarity_search(query, k=k)

def show(docs_list, title):
    print(f"\n=== {title} ===")
    for rank, d in enumerate(docs_list, 1):
        print(f"{rank}. [doc {d.metadata['doc_id']}] {d.metadata['chunk_id']}: {d.page_content[:120]}...")

for q in queries:
    for s_name, store in stores.items():
        hits = retrieve(store, q["query"])
        results[(q["qid"], s_name, "no_rerank")] = hits
        show(hits, f"{q['qid']} | {s_name} | NO rerank | target doc {q['target_doc']}")


=== Q1 | S1 | NO rerank | target doc 0 ===
1. [doc 0] s1-0-0: additive models provide an important family of models for semiparametric regression or classification. some reasons for ...
2. [doc 0] s1-0-10: fs and some results which can be interesting in their own. in this paper we provide some learning rates for the support ...
3. [doc 0] s1-0-2: unded influence function are known, see e.g. for the general case and for additive models. therefore, we will here consi...
4. [doc 0] s1-0-1: egularized kernel based models for additive models have been published when the focus is on sparsity and when the classi...
5. [doc 0] s1-0-4:  compare their risks evaluated for test data. for the same reason we will also not cover sparsity. consistency of suppor...

=== Q1 | S2 | NO rerank | target doc 0 ===
1. [doc 0] s2-0-1: in the last years many interesting results on learning rates of regularized kernel based models for additive models have...
2. [doc 0] s2-0-10: learning rates for quantile regre

### 5.2 Retrieval WITH reranking (cross-encoder)
Stage 1: vector search retrieves 25 candidates. Stage 2: the cross-encoder ms-marco-MiniLM-L-6-v2 scores each (query, chunk) pair jointly, and the top 5 are kept.

In [14]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda")
FETCH_K = 25

def retrieve_rerank(store, query, k=K, fetch_k=FETCH_K):
    candidates = store.similarity_search(query, k=fetch_k)
    scores = reranker.predict([(query, d.page_content) for d in candidates])
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:k]]

for q in queries:
    for s_name, store in stores.items():
        hits = retrieve_rerank(store, q["query"])
        results[(q["qid"], s_name, "rerank")] = hits
        show(hits, f"{q['qid']} | {s_name} | WITH rerank | target doc {q['target_doc']}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


=== Q1 | S1 | WITH rerank | target doc 0 ===
1. [doc 0] s1-0-10: fs and some results which can be interesting in their own. in this paper we provide some learning rates for the support ...
2. [doc 0] s1-0-2: unded influence function are known, see e.g. for the general case and for additive models. therefore, we will here consi...
3. [doc 0] s1-0-0: additive models provide an important family of models for semiparametric regression or classification. some reasons for ...
4. [doc 0] s1-0-21: ons on the goodness of our learning rates with those from the literature. as already mentioned in the introduction, some...
5. [doc 0] s1-0-28: rder hence the learning rate in theorem [quantilethm] is better than the one in (*??? 4.12) in this situation, if provid...

=== Q1 | S2 | WITH rerank | target doc 0 ===
1. [doc 0] s2-0-1: in the last years many interesting results on learning rates of regularized kernel based models for additive models have...
2. [doc 0] s2-0-30: of course, higher values of

In [15]:
rows = []
for q in queries:
    for s_name in stores:
        before = [d.metadata["chunk_id"] for d in results[(q["qid"], s_name, "no_rerank")]]
        after  = [d.metadata["chunk_id"] for d in results[(q["qid"], s_name, "rerank")]]
        rows.append({
            "query": q["qid"], "strategy": s_name,
            "top-1 before": before[0], "top-1 after": after[0],
            "kept": len(set(before) & set(after)),
            "new from reranking": ", ".join(c for c in after if c not in before) or "-",
        })
display(pd.DataFrame(rows))

,query,strategy,top-1 before,top-1 after,kept,new from reranking
0,Q1,S1,s1-0-0,s1-0-10,3,"s1-0-21, s1-0-28"
1,Q1,S2,s2-0-1,s2-0-1,3,"s2-0-30, s2-0-23"
2,Q2,S1,s1-3-24,s1-3-1,3,"s1-3-1, s1-3-23"
3,Q2,S2,s2-3-25,s2-3-1,4,s2-3-23
4,Q3,S1,s1-28-13,s1-28-2,4,s1-28-12
5,Q3,S2,s2-28-14,s2-28-3,4,s2-28-10
6,Q4,S1,s1-32-5,s1-32-0,3,"s1-32-37, s1-32-36"
7,Q4,S2,s2-32-4,s2-32-0,4,s2-32-39


### Retrieval observations
- Without reranking, all top-5 results already come from the correct paper, but several are generic
- Reranking reorders the results and pulls in chunks from deeper in the candidate list that more directly answer the query

## Step 6: Conversational Memory
A simple memory buffer stores previous user turns. Before retrieval, a follow-up query is rewritten by prepending the recent conversation history, so pronouns like "it" / "they" are resolved by the earlier context.

In [16]:
class ConversationMemory:
    def __init__(self, max_turns=2):
        self.turns = []              # past user queries
        self.max_turns = max_turns   # how many past turns to remember

    def add(self, query):
        self.turns.append(query)

    def rewrite(self, query):
        if not self.turns:
            return query
        history = " ".join(self.turns[-self.max_turns:])
        return f"{history} {query}"

    def clear(self):
        self.turns = []

# Multi-turn conversations: turn 2 cannot be understood without turn 1
conversations = [
    {"cid": "C1", "target_doc": 99, "turns": [
        "What is a network-on-chip based turbo decoder architecture?",
        "Which network topologies did they find work best for it?",
    ]},
    {"cid": "C2", "target_doc": 58, "turns": [
        "What are galled phylogenetic networks?",
        "What upper bounds do they prove for them?",
    ]},
]

memory = ConversationMemory()
for c in conversations:
    memory.clear()
    memory.add(c["turns"][0])
    print(f"{c['cid']} turn 2 original : {c['turns'][1]}")
    print(f"{c['cid']} turn 2 rewritten: {memory.rewrite(c['turns'][1])}\n")

C1 turn 2 original : Which network topologies did they find work best for it?
C1 turn 2 rewritten: What is a network-on-chip based turbo decoder architecture? Which network topologies did they find work best for it?

C2 turn 2 original : What upper bounds do they prove for them?
C2 turn 2 rewritten: What are galled phylogenetic networks? What upper bounds do they prove for them?



In [17]:
mem_results = {}
methods = {"no_rerank": retrieve, "rerank": retrieve_rerank}

for c in conversations:
    t1, t2 = c["turns"]
    for s_name, store in stores.items():
        for m_name, fn in methods.items():
            # WITHOUT memory
            mem_results[(c["cid"], s_name, m_name, "no_memory")] = fn(store, t2)

            # WITH memory
            memory.clear()
            fn(store, t1)
            memory.add(t1)
            mem_results[(c["cid"], s_name, m_name, "memory")] = fn(store, memory.rewrite(t2))


rows = []
for (cid, s_name, m_name, mem), hits in mem_results.items():
    target = next(c["target_doc"] for c in conversations if c["cid"] == cid)
    doc_ids = [d.metadata["doc_id"] for d in hits]
    rows.append({"conv": cid, "strategy": s_name, "method": m_name, "memory": mem,
                 "top-5 doc ids": doc_ids, "from target paper": f"{doc_ids.count(target)}/5"})
display(pd.DataFrame(rows))


show(mem_results[("C1", "S2", "rerank", "no_memory")], "C1 turn 2 | S2 | rerank | WITHOUT memory")
show(mem_results[("C1", "S2", "rerank", "memory")],    "C1 turn 2 | S2 | rerank | WITH memory")

,conv,strategy,method,memory,top-5 doc ids,from target paper
0,C1,S1,no_rerank,no_memory,"[99, 99, 99, 99, 99]",5/5
1,C1,S1,no_rerank,memory,"[99, 99, 99, 99, 99]",5/5
2,C1,S1,rerank,no_memory,"[99, 99, 99, 99, 99]",5/5
3,C1,S1,rerank,memory,"[99, 99, 99, 99, 99]",5/5
4,C1,S2,no_rerank,no_memory,"[99, 99, 99, 7, 99]",4/5
5,C1,S2,no_rerank,memory,"[99, 99, 99, 99, 99]",5/5
6,C1,S2,rerank,no_memory,"[99, 99, 99, 99, 99]",5/5
7,C1,S2,rerank,memory,"[99, 99, 99, 99, 99]",5/5
8,C2,S1,no_rerank,no_memory,"[5, 44, 65, 5, 5]",0/5
9,C2,S1,no_rerank,memory,"[58, 58, 58, 58, 58]",5/5



=== C1 turn 2 | S2 | rerank | WITHOUT memory ===
1. [doc 99] s2-99-10: 2. the routing memory overhead of the asp-ft algorithm (see fig. [fig: node] (b)) becomes relevant as decreases and ssp ...
2. [doc 99] s2-99-11: results referred to the same value are bounded into the same box and a label is assigned to each point to highlight the ...
3. [doc 99] s2-99-14: in this work a general framework to design network on chip based turbo decoder architectures has been presented. the pro...
4. [doc 99] s2-99-13: however, the solution with the smallest area is the one obtained with. the throughput flattening of low topologies can b...
5. [doc 99] s2-99-3: in,, ring, chordal ring and random graph topologies are investigated whereas in previous works are extended to mesh and ...

=== C1 turn 2 | S2 | rerank | WITH memory ===
1. [doc 99] s2-99-14: in this work a general framework to design network on chip based turbo decoder architectures has been presented. the pro...
2. [doc 99] s2-99-2: in orde

### Memory observations
| Conversation | Turn 2 | From target paper (no memory) | (with memory) |
|---|---|---|---|
| C1 (doc 99) | "Which network topologies did they find work best for it?" | 4–5/5 | 5/5 |
| C2 (doc 58) | "What upper bounds do they prove for them?" | 0–1/5 | 5/5 |

- **C2:** turn 2 is ambiguous on its own and retrieves unrelated papers (docs 5, 44, 65, 19). Memory resolves "them" → galled phylogenetic networks: **0/5 → 5/5**.
- **C1:** turn 2 contains a distinctive term ("network topologies") that only appears in doc 99, so it finds the right paper even without memory. With memory, the prepended general question shifted results toward overview chunks (paper structure, references) and away from topology-specific ones.
- **Takeaway:** simple history concatenation helps most when the follow-up is ambiguous, but can dilute an already-specific follow-up. An LLM-based query rewriter would avoid this.

## Step 7: Evaluation
**Relevance:** a chunk is relevant (1) if it comes from the target paper AND contains information useful for answering the query; otherwise 0. Labels are first suggested by a keyword rule, then manually reviewed and corrected.
**Recall@K denominator:** pooling. The relevant set for a query is all relevant chunks retrieved by any method for that query and chunking strategy.

In [18]:
# target paper for every evaluated query (single-turn + turn 2 of each conversation)
target_doc = {q["qid"]: q["target_doc"] for q in queries}
target_doc.update({c["cid"]: c["target_doc"] for c in conversations})


answer_terms = {
    "Q1": [["learning rate", "rate of convergence", "rates of convergence"]],
    "Q2": [["scintillation"], ["diffuser", "phase"]],
    "Q3": [["correlated"], ["encod", "codeword", "recover"]],
    "Q4": [["od matri", "origin-destination"], ["survey", "count"]],
    "C1": [["topolog"], ["kautz", "bruijn", "throughput", "suited"]],
    "C2": [["reticulation"], ["at most", "bound"]],
}


all_runs = {}
for (qid, s, m), hits in results.items():
    all_runs[(qid, s, m)] = hits
for (cid, s, m, mem), hits in mem_results.items():
    all_runs[(cid, s, f"{m}|{mem}")] = hits


pool = {}
for (qid, s, cfg), hits in all_runs.items():
    for d in hits:
        pool.setdefault((qid, s), {})[d.metadata["chunk_id"]] = d

def suggest_label(qid, d):
    if d.metadata["doc_id"] != target_doc[qid]:
        return 0
    text = d.page_content.lower()
    return int(all(any(t in text for t in group) for group in answer_terms[qid]))

suggested = {}
for (qid, s), chunks in pool.items():
    for cid, d in chunks.items():
        suggested[(qid, cid)] = suggest_label(qid, d)

print("total runs:", len(all_runs))
print("unique chunks to label:", len(suggested))
print("suggested relevant:", sum(suggested.values()))

total runs: 32
unique chunks to label: 101
suggested relevant: 64


In [21]:
import textwrap

def review(qid):
    print(f"##### {qid} | target doc {target_doc[qid]} #####")
    for (q, s), chunks in pool.items():
        if q != qid:
            continue
        for cid, d in chunks.items():
            print(f"\n[{cid}] doc {d.metadata['doc_id']} | suggested = {suggested[(qid, cid)]}")
            print(textwrap.fill(d.page_content, 120))

In [34]:
overrides = {
    # Q1: background / setup chunks that do not state learning-rate results
    ("Q1", "s1-0-0"): 0, ("Q1", "s1-0-2"): 0, ("Q1", "s1-0-4"): 0,
    ("Q1", "s2-0-1"): 0, ("Q1", "s2-0-4"): 0,

    # Q2: purpose/formalism chunks -> 0; mechanism chunks the keyword rule missed -> 1
    ("Q2", "s1-3-3"): 0, ("Q2", "s1-3-4"): 0, ("Q2", "s2-3-4"): 0,
    ("Q2", "s1-3-15"): 1, ("Q2", "s1-3-1"): 1, ("Q2", "s2-3-1"): 1,

    # Q3: noise definition only -> 0; chunk stating the encoding result -> 1
    ("Q3", "s2-28-3"): 0, ("Q3", "s1-28-3"): 1,

    # Q4: intro / future-work chunks -> 0; chunks describing counts + survey inputs -> 1
    ("Q4", "s1-32-0"): 0, ("Q4", "s2-32-0"): 0, ("Q4", "s1-32-39"): 0,
    ("Q4", "s1-32-36"): 1, ("Q4", "s2-32-11"): 1,

    # C1: area breakdown, paper roadmap, prior-work topologies -> 0
    ("C1", "s1-99-13"): 0, ("C1", "s2-99-13"): 0, ("C1", "s1-99-4"): 0,
    ("C1", "s2-99-4"): 0, ("C1", "s1-99-3"): 0, ("C1", "s2-99-3"): 0,

    # C2: bounds for a different network class (reticulation visible) or empty table -> 0
    ("C2", "s1-58-12"): 0, ("C2", "s2-58-12"): 0, ("C2", "s2-58-64"): 0,
}

# final labels = keyword suggestion, corrected by manual review
labels = {**suggested, **overrides}

typos = [k for k in overrides if k not in suggested]
print("manual corrections:", len(overrides))
print("typos (should be empty):", typos)
print("final relevant chunks:", sum(labels.values()), "of", len(labels))

manual corrections: 27
typos (should be empty): []
final relevant chunks: 49 of 101


### Manual relevance review
All 101 pooled chunks were read and judged against the question. The keyword rule was corrected in **27 cases**: it over-counted background, roadmap, related-work and intro chunks that mention the keywords but do not answer the question, and under-counted chunks that explain the answer without the exact keywords (e.g. Q2 chunks describing the diffuser mechanism without the word "scintillation"). Each correction is listed with its reason in the overrides cell above.

### Metrics (K = 5)
- **Precision@5** = relevant chunks in top 5 / 5
- **Recall@5** = relevant chunks in top 5 / total relevant chunks in the pool (per query and strategy)
- **MRR** = mean of 1 / rank of the first relevant chunk (0 if none in top 5)

In [22]:
review("Q2")

##### Q2 | target doc 3 #####

[s1-3-24] doc 3 | suggested = 1
ntities become independent of at long propagation paths because light scattering on atmospheric inhomogeneities prevails
in this case. the plots in figs. 3 anf 4 show that the finite-time effect is quite sizable even for very slow "
detectors (). our paper makes it possible to estimate the actual utility of phase diffusers in several physical regimes.
we have analyzed the effects of a diffuser on scintillations for the case of large-amplitude phase fluctuations. this
specific case is very convenient for theoretial analysis because only two parameters are required to describe the
effects of the diffuser. phase fluctuations may occur independently in space as well as in time. also, our formalism can
be applied for the physical situation in which a spatially random phase distribution drifts a

[s1-3-25] doc 3 | suggested = 1
sm can be applied for the physical situation in which a spatially random phase distribution drifts acro

In [24]:
review("Q3")

##### Q3 | target doc 28 #####

[s1-28-13] doc 28 | suggested = 1
 irrespective of error operators acted in the channel.,width=529] we have shown that there is a quantum error correction
which suppresses fully correlated errors of the form, in which qubits are required to encode (i) data qubit states when
is odd and (ii) data qubit states when is even. we have proved these statements by using operator theoretical technique.
neither syndrome measurements nor ancilla qubits for syndrome measurement are required in our scheme, which makes
physical implementation of our scheme highly practical. examples with and are analyzed in detail and explicit quantum
circuits implementing our qec with the least number of cnot gate were obtained. since the error operators are closed
under matrix multiplication, errors can be corrected even when they act on the codeword m

[s1-28-14] doc 28 | suggested = 1
are closed under matrix multiplication, errors can be corrected even when they act on the codeword

In [26]:
review("Q4")

##### Q4 | target doc 32 #####

[s1-32-5] doc 32 | suggested = 1
ume that the behaviour of car-user and public transport users are the same, at least regarding their respective od
matrix. given that the public transport network topology is often different from the road network topology, one may
doubt the accuracy of this assumption. moreover, they just use the partial structure extracted from the surveys. + the
purpose of this paper is then to develop an estimation procedure for the origin-destination matrix based on the ticket
records available for the transport network and/or on previous surveys. unlike the article from bierliaire, we use
survey data collection from public transport users, and estimate the approximate whole matrix structure through the
estimation of its eigenvectors. we propose a robust version of the estimator to avoid biases induced

[s1-32-4] doc 32 | suggested = 1
 using survey data. indeed, if no complete information is available about the passengers destination

In [30]:
for cid, d in pool[("C1", "S2")].items():
    print(f"\n[{cid}] doc {d.metadata['doc_id']} | suggested = {suggested[('C1', cid)]}")
    print(textwrap.fill(d.page_content, 120))


[s2-99-11] doc 99 | suggested = 1
results referred to the same value are bounded into the same box and a label is assigned to each point to highlight the
corresponding topology, namely topologies are identified as r-ring, h-honeycomb, t-toroidal mesh, k-generalized kautz
with the corresponding value (k2, k3, k4). as it can be observed, generalized kautz topologies with (k4) are always the
best solutions to achieve high throughput with minimum area overhead. in fig. [fig: tar_tot] significant results
extracted from table [tab: wimax_results] and [tab: mhoms_results] are shown in graphical form. in particular, for the
asp-ft routing algorithm is the best solution, whereas for ssp routing algorithms, implemented as in fig.

[s2-99-13] doc 99 | suggested = 1
however, the solution with the smallest area is the one obtained with. the throughput flattening of low topologies can
be explained by observing that high values of tend to saturate the network. furthermore, high values of lengthen th

In [33]:
for cid, d in pool[("C2", "S2")].items():
    print(f"\n[{cid}] doc {d.metadata['doc_id']} | suggested = {suggested[('C2', cid)]}")
    print(textwrap.fill(d.page_content, 120))


[s2-44-48] doc 44 | suggested = 0
using the bounds which are valid for all integers and with, we easily find that since and as, while,, and stay bounded,
the desired claim follows by taking sufficiently large. the reader may have noticed that, in most of our previous
arguments, and were restricted to be away from the boundary values. to control the situation near the boundary values,
we have to prove the following claim: [lemma3.5] for each and each, let be the event then for each and each there exists
an such that proof we will split the complement of into four events and prove the corresponding estimate for each of
them. we begin with the event. the main tool will be stochastic domination by a product measure. consider the usual
partial order on spin configurations defined by putting whenever for all.

[s2-5-21] doc 5 | suggested = 0
[fig: interp], width=340] [ajpositive] 1. [a3l] for all 2. [a5l] for all 3. [a7l] for all 4. [a9l] for all all the
stated bounds are determined by inte

In [35]:
def evaluate(qid, s_name, hits, k=K):
    rels = [labels[(qid, d.metadata["chunk_id"])] for d in hits[:k]]
    total_rel = sum(labels[(qid, cid)] for cid in pool[(qid, s_name)])
    precision = sum(rels) / k
    recall = sum(rels) / total_rel if total_rel else 0.0
    rr = next((1 / (i + 1) for i, r in enumerate(rels) if r), 0.0)
    return precision, recall, rr, rels

rows = []
for (qid, s_name, cfg), hits in all_runs.items():
    p, r, rr, rels = evaluate(qid, s_name, hits)
    rerank = "rerank" if cfg.startswith("rerank") else "no_rerank"
    memory = cfg.split("|")[1] if "|" in cfg else "-"
    rows.append({"query": qid, "strategy": s_name, "rerank": rerank, "memory": memory,
                 "labels top5": rels, "P@5": p, "R@5": round(r, 3), "RR": round(rr, 3)})

eval_df = pd.DataFrame(rows)
display(eval_df)

,query,strategy,rerank,memory,labels top5,P@5,R@5,RR
0,Q1,S1,no_rerank,-,"[0, 1, 0, 0, 0]",0.2,0.333,0.5
1,Q1,S2,no_rerank,-,"[0, 1, 1, 0, 0]",0.4,0.500,0.5
2,Q2,S1,no_rerank,-,"[1, 1, 0, 1, 0]",0.6,0.600,1.0
3,Q2,S2,no_rerank,-,"[1, 1, 1, 0, 1]",0.8,0.800,1.0
4,Q3,S1,no_rerank,-,"[1, 1, 1, 0, 1]",0.8,0.800,1.0
5,Q3,S2,no_rerank,-,"[1, 0, 1, 1, 0]",0.6,0.750,1.0
6,Q4,S1,no_rerank,-,"[1, 1, 0, 0, 0]",0.4,0.500,1.0
7,Q4,S2,no_rerank,-,"[1, 1, 0, 0, 1]",0.6,0.750,1.0
8,Q1,S1,rerank,-,"[1, 0, 0, 1, 1]",0.6,1.000,1.0
9,Q1,S2,rerank,-,"[0, 1, 1, 1, 1]",0.8,1.000,0.5


In [36]:
metrics = ["P@5", "R@5", "RR"]
single = eval_df[eval_df["memory"] == "-"]
multi  = eval_df[eval_df["memory"] != "-"]

def summarize(df, by):
    out = df.groupby(by)[metrics].mean().round(3)
    return out.rename(columns={"RR": "MRR"})

print("Comparison 1: WITHOUT vs WITH reranking (Q1–Q4, both strategies)")
display(summarize(single, "rerank"))

print("Comparison 2: Chunking S1 vs S2 (Q1–Q4, both retrieval methods)")
display(summarize(single, "strategy"))

print("Full grid: strategy x reranking (Q1–Q4)")
display(summarize(single, ["strategy", "rerank"]))

print("Comparison 3: WITHOUT vs WITH memory (turn 2 of C1, C2)")
display(summarize(multi, "memory"))

print("Memory per conversation")
display(summarize(multi, ["query", "memory"]))

Comparison 1: WITHOUT vs WITH reranking (Q1–Q4, both strategies)


,P@5,R@5,MRR
rerank,,,
no_rerank,0.550,0.629,0.875
rerank,0.775,0.919,0.750


Comparison 2: Chunking S1 vs S2 (Q1–Q4, both retrieval methods)


,P@5,R@5,MRR
strategy,,,
S1,0.625,0.729,0.875
S2,0.700,0.819,0.750


Full grid: strategy x reranking (Q1–Q4)


P@5    R@5    MRR
strategy rerank                       
S1       no_rerank  0.50  0.558  0.875
         rerank     0.75  0.900  0.875
S2       no_rerank  0.60  0.700  0.875
         rerank     0.80  0.938  0.625

Comparison 3: WITHOUT vs WITH memory (turn 2 of C1, C2)


,P@5,R@5,MRR
memory,,,
memory,0.400,0.529,0.875
no_memory,0.275,0.396,0.438


Memory per conversation


P@5    R@5    MRR
query memory                       
C1    memory     0.20  0.292  1.000
      no_memory  0.55  0.792  0.875
C2    memory     0.60  0.767  0.750
      no_memory  0.00  0.000  0.000

## Step 8: Findings and Comparison

### 1. Without vs with reranking (Q1–Q4)
| | P@5 | R@5 | MRR |
|---|---|---|---|
| No reranking | 0.550 | 0.629 | 0.875 |
| With reranking | **0.775** | **0.919** | 0.750 |

Reranking gave the largest improvement: **precision +41%, recall +46%**. The cross-encoder reads the query and chunk together, so it pulled useful chunks from ranks 6–25 (e.g. Q1 chunks on convergence rates) and pushed generic ones out of the top 5.
However, **MRR dropped** (0.875 → 0.750), entirely in S2 (0.875 → 0.625): the reranker placed background chunks at rank 1 that restate the query topic without answering it (Q1: related-work chunk, Q3: noise definition, Q4: generic OD-matrix intro). The `ms-marco` reranker is trained on web search, so it rewards topical overlap more than "answers the question".

### 2. Chunking strategy 1 vs 2 (Q1–Q4)
| | P@5 | R@5 | MRR |
|---|---|---|---|
| S1: Fixed-size | 0.625 | 0.729 | 0.875 |
| S2: Sentence-aware | **0.700** | **0.819** | 0.750 |

Sentence-aware chunking was better on precision and recall, both with and without reranking (no rerank P@5 0.50 → 0.60; rerank 0.75 → 0.80). 96.7% of S2 chunks end at a sentence boundary vs 3.5% for S1, so key statements are not split across chunks (e.g. in Q3 a result sentence was cut off in S1). The lower S2 MRR comes only from the reranker interaction above; without reranking both strategies have MRR 0.875.
**Best configuration: S2 + reranking (P@5 0.80, R@5 0.938).**

### 3. Without vs with conversational memory (turn 2 of C1, C2)
| | P@5 | R@5 | MRR |
|---|---|---|---|
| No memory | 0.275 | 0.396 | 0.438 |
| With memory | **0.400** | **0.529** | **0.875** |

Memory improved all three metrics on average, but the effect depends on the follow-up:
- **C2** ("What upper bounds do they prove for them?") is ambiguous on its own and retrieved only unrelated papers: **P@5 0.00 → 0.60**, MRR 0.00 → 0.75.
- **C1** ("Which network topologies did they find work best for it?") already contains a distinctive term, so it worked without memory. Prepending turn 1 diluted the query toward overview chunks: **P@5 0.55 → 0.20** (MRR still 1.0).

Simple history concatenation resolves pronouns but can drown out a specific follow-up. An LLM-based query rewriter, or applying memory only when the query is ambiguous, would avoid this.

### Overall conclusions
1. **Reranking** has the biggest impact on precision and recall.
2. **Sentence-aware chunking** is consistently better than fixed-size chunking.
3. **Conversational memory** is essential for ambiguous follow-ups but should be applied selectively.

### Limitations
- Small evaluation set (4 single-turn queries, 2 conversations) at K = 5; results are indicative, not statistically significant.
- Relevance was judged manually by one person; Recall@K uses a pooled relevant set (relevant chunks found by any method), so it is relative recall.
- The dataset replaces math and some names with `@xmath`/`@xcite` placeholders, so some answers (e.g. exact bounds in C2) are incomplete in every chunk.